<a href="https://colab.research.google.com/github/lolismek/AroTranslate/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentencepiece transformers==4.33 datasets sacremoses sacrebleu  -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 46.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.7/536.7 kB 47.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 60.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.3/106.3 kB 13.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 103.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 18.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
import os
if not os.path.exists('/gd'):
    drive.mount('/gd')
import locale
def gpe(x=None):
    return "UTF-8"
locale.getpreferredencoding = gpe

Mounted at /gd


In [ ]:
import pandas as pd
pd.options.display.max_colwidth = 100
trans_df = pd.read_csv('/gd/MyDrive/transfer/data.csv', dtype={'rup': str, 'ron': str})
df_train = trans_df[trans_df.split=='train'].copy()
df_dev = trans_df[trans_df.split=='dev'].copy()
df_test = trans_df[trans_df.split=='test'].copy()

In [ ]:
from transformers import NllbTokenizer
from tqdm.auto import tqdm, trange
tokenizer = NllbTokenizer.from_pretrained('facebook/nllb-200-distilled-600M')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.55k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

In [ ]:
# this code is adapted from  the Stopes repo of the NLLB team
# https://github.com/facebookresearch/stopes/blob/main/stopes/pipelines/monolingual/monolingual_line_processor.py#L214

import re
import sys
import typing as tp
import unicodedata
from sacremoses import MosesPunctNormalizer


mpn = MosesPunctNormalizer(lang="en")
mpn.substitutions = [
    (re.compile(r), sub) for r, sub in mpn.substitutions
]


def get_non_printing_char_replacer(replace_by: str = " ") -> tp.Callable[[str], str]:
    non_printable_map = {
        ord(c): replace_by
        for c in (chr(i) for i in range(sys.maxunicode + 1))
        # same as \p{C} in perl
        # see https://www.unicode.org/reports/tr44/#General_Category_Values
        if unicodedata.category(c) in {"C", "Cc", "Cf", "Cs", "Co", "Cn"}
    }

    def replace_non_printing_char(line) -> str:
        return line.translate(non_printable_map)

    return replace_non_printing_char

replace_nonprint = get_non_printing_char_replacer(" ")

def preproc(text):
    clean = mpn.normalize(text)
    clean = replace_nonprint(clean)
    # replace 𝓕𝔯𝔞𝔫𝔠𝔢𝔰𝔠𝔞 by Francesca
    clean = unicodedata.normalize("NFKC", clean)
    return clean

In [ ]:
texts_with_unk = [text for text in tqdm(trans_df.rup) if tokenizer.unk_token_id in tokenizer(preproc(text)).input_ids]
print(len(texts_with_unk))

  0%|          | 0/73509 [00:00<?, ?it/s]

0


In [ ]:
from transformers import AutoModelForSeq2SeqLM
from transformers import NllbTokenizer
tokenizer = NllbTokenizer.from_pretrained('facebook/nllb-200-distilled-600M')
print(len(tokenizer))
#print(tokenizer.convert_tokens_to_ids(['ron_Latn', '<mask>'])) # 256145, 256203
print(tokenizer.convert_ids_to_tokens([256145, 256203]))

256204
['ron_Latn', '<mask>']


In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained('facebook/nllb-200-distilled-600M')
model.resize_token_embeddings(len(tokenizer))

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

You are resizing the embedding layer without providing a `pad_to_multiple_of` parameter. This means that the new embedding dimension will be 256204. This might induce some performance reduction as *Tensor Cores* will not be available. For more details about this, or help on choosing the correct value for resizing, refer to this guide: https://docs.nvidia.com/deeplearning/performance/dl-performance-matrix-multiplication/index.html#requirements-tc


Embedding(256204, 1024)

In [ ]:
model_load_name = '/gd/MyDrive/transfer/models/ron-rup-v2'
model = AutoModelForSeq2SeqLM.from_pretrained(model_load_name).cuda()
tokenizer = NllbTokenizer.from_pretrained(model_load_name)

In [ ]:
import gc
import random
import numpy as np
import torch
from tqdm.auto import tqdm, trange
from transformers.optimization import Adafactor
from transformers import get_constant_schedule_with_warmup

def cleanup():
    """Try to free GPU memory"""
    gc.collect()
    torch.cuda.empty_cache()

cleanup()
model.cuda()

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): Embedding(256204, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): Embedding(256204, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_norm): LayerNorm

In [ ]:
optimizer = Adafactor(
    [p for p in model.parameters() if p.requires_grad],
    scale_parameter=False,
    relative_step=False,
    lr=1e-4,
    clip_threshold=1.0,
    weight_decay=1e-3,
)
batch_size = 16  # 32 already doesn't fit well to 15GB of GPU memory
max_length = 128
warmup_steps = 1_000
training_steps = 20000

losses = []
scheduler = get_constant_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps)

In [ ]:
LANGS = [('ron', 'ron_Latn'), ('rup', 'ron_Latn')]

def get_batch_pairs(batch_size, data=df_train):
    (l1, long1), (l2, long2) = random.sample(LANGS, 2)
    xx, yy = [], []
    for _ in range(batch_size):
        item = data.iloc[random.randint(0, len(data)-1)]
        xx.append(preproc(item[l1]))
        yy.append(preproc(item[l2]))
    return xx, yy, long1, long2

print(get_batch_pairs(2))


(['israelitslji tuts, cara-l videa omlu-atsel, fudzea di el, shi multu s-asparea;', 'ghini ma, al pitricushi tini duhlu-a tat'], ['toti israelitii, vazand pe omul acela, fugeau de el, temandu-se foarte tare;', 'doamne, cine este asemenea tie intre dumnezei? cine este asemenea tie preaslavit in sfintenie, minunat intru slava si facator de minuni?'], 'ron_Latn', 'ron_Latn')


In [ ]:
MODEL_SAVE_PATH = '/gd/MyDrive/transfer/models/ron-rup-v2'

In [ ]:
def train_model():
  model.train()
  x, y, loss = None, None, None
  cleanup()

  tq = trange(len(losses), training_steps)
  for i in tq:
      xx, yy, lang1, lang2 = get_batch_pairs(batch_size)
      try:
          tokenizer.src_lang = lang1
          x = tokenizer(xx, return_tensors='pt', padding=True, truncation=True, max_length=max_length).to(model.device)
          tokenizer.src_lang = lang2
          y = tokenizer(yy, return_tensors='pt', padding=True, truncation=True, max_length=max_length).to(model.device)
          y.input_ids[y.input_ids == tokenizer.pad_token_id] = -100

          loss = model(**x, labels=y.input_ids).loss
          loss.backward()
          losses.append(loss.item())

          optimizer.step()
          optimizer.zero_grad(set_to_none=True)
          scheduler.step()

      except RuntimeError as e:
          optimizer.zero_grad(set_to_none=True)
          x, y, loss = None, None, None
          cleanup()
          print('error', max(len(s) for s in xx + yy), e)
          continue

      if i % 1000 == 0:
          print(i, np.mean(losses[-1000:]))

      if i % 1000 == 0 and i > 0:
          model.save_pretrained(MODEL_SAVE_PATH)
          tokenizer.save_pretrained(MODEL_SAVE_PATH)

In [ ]:
# RESUME LEARNING (if the case):
# model_load_name = '/gd/MyDrive/transfer/models/ron-rup-v2'
# model = AutoModelForSeq2SeqLM.from_pretrained(model_load_name).cuda()
# tokenizer = NllbTokenizer.from_pretrained(model_load_name)
train_model() # epochs (1000 training steps): 2 + 20: 0.86
# + 16 / 0.66
#

  0%|          | 0/20000 [00:00<?, ?it/s]

0 0.5205789804458618
1000 0.8387558807134629
2000 0.8358263720571995
3000 0.8236296298503876
4000 0.8177344238758087
5000 0.7932549526691437
6000 0.7867979174256324
7000 0.7732649122476578
8000 0.7643391852378845
9000 0.7332970747053623
10000 0.7322154013812542
11000 0.7187940168976784
12000 0.704498915463686
13000 0.7016019575893879
14000 0.6665872095376253
15000 0.65933497184515
16000 0.6656051858067512


KeyboardInterrupt: 

In [ ]:
def translate(text, src_lang='ron_Latn', tgt_lang='ron_Latn', a=32, b=3, max_input_length=1024, num_beams=4, **kwargs):
    tokenizer.src_lang = src_lang
    tokenizer.tgt_lang = tgt_lang
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True, max_length=max_input_length)
    result = model.generate(
        **inputs.to(model.device),
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
        max_new_tokens=int(a + b * inputs.input_ids.shape[1]),
        num_beams=num_beams,
        **kwargs
    )
    return tokenizer.batch_decode(result, skip_special_tokens=True)

In [ ]:
t = "ti-a oaminjlor pidimolu. sh-mash mini."
print(translate(t, 'ron_Latn', 'ron_Latn'))

['pentru oamenilor pedeapsa, si numai pentru mine.']


In [ ]:
def batched_translate(texts, batch_size=16, **kwargs):
    """Translate texts in batches of similar length"""
    idxs, texts2 = zip(*sorted(enumerate(texts), key=lambda p: len(p[1]), reverse=True))
    results = []
    for i in tqdm(trange(0, len(texts2), batch_size)):
        results.extend(translate(texts2[i: i+batch_size], **kwargs))
    return [p for i, p in sorted(zip(idxs, results))]

In [ ]:
import sacrebleu

bleu_calc = sacrebleu.BLEU()

translated = batched_translate(df_dev.rup.values)
reals = df_dev.ron.to_list()
print(translated)
print(reals)
print(bleu_calc.corpus_score(translated, reals))

# sampl = trans_df.sample(1000)
# translated = batched_translate(sampl.rup.values)
# reals = sampl.ron.to_list()
# print(bleu_calc.corpus_score(translated, reals))



  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

['pizmui', 'ureu', 'imparatul.', 'gratiie', 'barbatul intelept se va umple de binecuvantare si va veseli pe toti cei ce-l vor vedea,', 'spulbiru', 'imprima', 'templul, pe care l-a zidit regele solomon pentru domnul era lung de saizeci de coti, lat de douazeci si inalt de treizeci de coti.', 'si a luat balac pe valaam in varful lui peor, care se intoarse spre pustie.', 'jumatate', 'greier', 'daca te va intreba in viitor fiul tau si va zice: ce inseamna aceste porunci, hotarari si legile pe care vi le-a dat domnul dumnezeul vostru? sa spui fiului tau:', 'scop', 'tom tacu o clipa, apoi intreba:', "- sora, da' ninti, ca vin si eu,", 'si a grait domnul cu moise si a zis:', 'vatra', 'cantec', 'groaza te voi face si nu vei mai fi; te vor cauta si nu te vor afla in veac, zice domnul dumnezeu.', 'amarea!, amarea!', 'pe-agale, pe-agale s-au departat', 'de cand erai la plete, nu erai balos.', 'dispozitie', 'si a facut, in latura de dedesubt a templului, o platura de douazeci de coti in lungime si

In [ ]:
chrf_calc = sacrebleu.CHRF(word_order=2)  # this metric is called ChrF++
print(chrf_calc.corpus_score(translated, reals))

chrF2++ = 22.73


In [ ]:
sampl = trans_df.sample(1000)
a = batched_translate(sampl.rup.values)
b = sampl.ron.to_list()

  0%|          | 0/63 [00:00<?, ?it/s]

  0%|          | 0/63 [00:00<?, ?it/s]

In [ ]:
print(a[504])
print(b[504])
print(chrf_calc.corpus_score(a, b))

fecior, aiurea
fecior ne-ai fost


NameError: name 'chrf_calc' is not defined

In [ ]:
c = sampl.rup.to_list()
df = pd.DataFrame({'rup': c, 'ron_translated': a, 'ron_refference': b})
print(df)

                                                                                                     rup  \
0                                                                                  diparti fugu di doru,   
1                                                                                             aynanghiu    
2                                                                                                clotsa    
3    dicara s-dishtipta noe dit amurtsatlu di yin shi cara-avdza tsi-lji featsi ficiorlu-a lui atsel ...   
4                                                                                              muldzara    
..                                                                                                   ...   
995                                                                                           ipucrisii    
996                                                                                         nascarsescu    
997                         

In [ ]:
df.to_csv('translations.csv')

In [ ]:
print(bleu_calc.corpus_score(translated, [reals]))


BLEU = 49.09 70.1/54.8/44.3/36.4 (BP = 0.984 ratio = 0.984 hyp_len = 4714 ref_len = 4791)


In [ ]:
x = batched_translate(df_test.rup.values)
y = df_test.ron.to_list()
print(bleu_calc.corpus_score(x, [y]))

df = pd.DataFrame({'rup': df_test.rup.to_list(), 'ron_translated': x, 'ron_refference': y})
df.to_csv('rup->ron_test.csv')

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

BLEU = 45.93 69.4/52.9/41.6/33.4 (BP = 0.967 ratio = 0.968 hyp_len = 4056 ref_len = 4192)


In [ ]:
x = batched_translate(df_test.ron.values)
y = df_test.rup.to_list()
print(bleu_calc.corpus_score(x, [y]))

df = pd.DataFrame({'ron': df_test.ron.to_list(), 'rup_translated': x, 'rup_refference': y})
df.to_csv('ron->rup_test.csv')

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

BLEU = 41.45 62.8/46.8/35.8/28.0 (BP = 1.000 ratio = 1.009 hyp_len = 3671 ref_len = 3640)
